### Generate the INPUT file for each GW events to use SNANA to simulate kilonova light curves.

In [174]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import re
import csv
import pprint

template_path = Path("/fred/oz016/bgao_kn/ML+GW+KN/dataset/KN_sim/SIM_INPUT/SIMGEN_KN_LSST_TEMPLATE.INPUT")
simlib_dir = "/fred/oz016/bgao_kn/data/simlib/"

In [175]:
injections = pd.read_csv("/fred/oz016/bgao_kn/ML+GW+KN/dataset/O5_sim_bns/injections_final.csv")
sim_id = 6


In [176]:
# get NLIBID from simlib file
def get_NLIBID(simlib_file):
    nlibid = []
    with open(simlib_file, 'r') as f:
        lines = f.readlines()
        for line in lines:
            if line.startswith('NLIBID:'):
                nlibid = int(line.split()[1])
                break
    return nlibid

NLIBID = get_NLIBID(os.path.join(simlib_dir, f"baseline_v5.0.1_10yrs_{sim_id}.SIMLIB"))
print(f"NLIBID from simlib: {NLIBID}")

NLIBID from simlib: 13107


In [177]:
template_text = template_path.read_text()
pprint.pprint(template_text)

('# Created November 2025\n'
 '\n'
 'SIMLIB_NREPEAT:   1 # use each LIBID for one time\n'
 'NGENTOT_LC: 13107  # number of LC to generate, number of LIBID in SIMLIB '
 'file\n'
 '\n'
 '# name of version to appear in $SNDATA_ROOT/SIM\n'
 'GENVERSION: MY_LSST_KN\n'
 '\n'
 '# define SED model and rate model\n'
 'GENMODEL:  '
 '$SNDATA_ROOT/models/SIMSED/SIMSED.KN-BULLA19/SIMSED.BULLA-BNS-M3-3COMP\n'
 'DNDZ: POWERLAW  1.0E-6  0.0  # volumetric rate = 1.0E-6/Mpc^3/yr (no '
 'z-dependence)\n'
 '\n'
 'GENFILTERS:   ugrizY\n'
 '\n'
 '# define distribution of KN parameters\n'
 'SIMSED_PARAM: COSTHETA\n'
 'GENPEAK_COSTHETA: 0.5\n'
 'GENRANGE_COSTHETA: 0.0 1.0\n'
 'GENSIGMA_COSTHETA: 0.01 0.01\n'
 '\n'
 'SIMSED_PARAM: MEJDYN\n'
 'GENPEAK_MEJDYN: 0.01\n'
 'GENRANGE_MEJDYN: 0.001 0.02\n'
 'GENSIGMA_MEJDYN: 0 0\n'
 '\n'
 'SIMSED_PARAM: MEJWIND\n'
 'GENPEAK_MEJWIND: 0.05\n'
 'GENRANGE_MEJWIND: 0.01 0.13\n'
 'GENSIGMA_MEJWIND: 0 0\n'
 '\n'
 'SIMSED_PARAM: PHI\n'
 'GENPEAK_PHI: 45\n'
 'GENRANGE_PHI: 0 

In [ ]:
# replace NLIBID and GENVERSION in the template
text = template_text
NGENTOT_LC_line = f"NGENTOT_LC: {NLIBID}"
text = re.sub(
        r"^(NGENTOT_LC:\s*)\S+.*$",
        rf"\1 {NLIBID}",
        text,
        flags=re.MULTILINE
)
genversion_new = f"MY_LSST_KN_{sim_id}"
text = re.sub(
    r"^(GENVERSION:\s*)\S+.*$",
    rf"\1{genversion_new}",
    text,
    flags=re.MULTILINE
)

('# Created November 2025\n'
 '\n'
 'SIMLIB_NREPEAT:   1 # use each LIBID for one time\n'
 'NGENTOT_LC:  13107\n'
 '\n'
 '# name of version to appear in $SNDATA_ROOT/SIM\n'
 'GENVERSION: MY_LSST_KN_6\n'
 '\n'
 '# define SED model and rate model\n'
 'GENMODEL:  '
 '$SNDATA_ROOT/models/SIMSED/SIMSED.KN-BULLA19/SIMSED.BULLA-BNS-M3-3COMP\n'
 'DNDZ: POWERLAW  1.0E-6  0.0  # volumetric rate = 1.0E-6/Mpc^3/yr (no '
 'z-dependence)\n'
 '\n'
 'GENFILTERS:   ugrizY\n'
 '\n'
 '# define distribution of KN parameters\n'
 'SIMSED_PARAM: COSTHETA\n'
 'GENPEAK_COSTHETA: 0.5\n'
 'GENRANGE_COSTHETA: 0.0 1.0\n'
 'GENSIGMA_COSTHETA: 0.01 0.01\n'
 '\n'
 'SIMSED_PARAM: MEJDYN\n'
 'GENPEAK_MEJDYN: 0.01\n'
 'GENRANGE_MEJDYN: 0.001 0.02\n'
 'GENSIGMA_MEJDYN: 0 0\n'
 '\n'
 'SIMSED_PARAM: MEJWIND\n'
 'GENPEAK_MEJWIND: 0.05\n'
 'GENRANGE_MEJWIND: 0.01 0.13\n'
 'GENSIGMA_MEJWIND: 0 0\n'
 '\n'
 'SIMSED_PARAM: PHI\n'
 'GENPEAK_PHI: 45\n'
 'GENRANGE_PHI: 0 90\n'
 'GENSIGMA_PHI: 1 1\n'
 '\n'
 '\n'
 '# define 3 quantit

In [179]:
idx = np.where(injections['simulation_id']==sim_id)[0]
mjd_explode = injections['mjd_time'].iloc[idx].values
costheta = injections['costheta'].iloc[idx].values
phi = injections['phi'].iloc[idx].values
mej_dyn = injections['mej_dyn'].iloc[idx].values
mej_wind = injections['mej_wind'].iloc[idx].values

# modify explosion time and ejecta parameters
text = re.sub(
    r"^(MJD_EXPLODE:\s*)\S+.*$",
    rf"\1 {mjd_explode[0]}",
    text,
    flags=re.MULTILINE
)
text = re.sub(
    r"^(GENPEAK_COSTHETA:\s*)\S+.*$",
    rf"\1 {costheta[0]}",
    text,
    flags=re.MULTILINE
)
text = re.sub(
    r"^(GENPEAK_PHI:\s*)\S+.*$",
    rf"\1 {phi[0]}",
    text,
    flags=re.MULTILINE
)
text = re.sub(
    r"^(GENPEAK_MEJDYN:\s*)\S+.*$",
    rf"\1 {mej_dyn[0]}",
    text,
    flags=re.MULTILINE
)
text = re.sub(
    r"^(GENPEAK_MEJWIND:\s*)\S+.*$",
    rf"\1 {mej_wind[0]}",
    text,
    flags=re.MULTILINE
)
# MODIFY SIMLIB FILE
text = re.sub(
    r"^(SIMLIB_FILE:\s*.+)_(?=\.SIMLIB)",
    fr"\1_{sim_id}",
    text,
    flags=re.MULTILINE
)

pprint.pprint(text)

('# Created November 2025\n'
 '\n'
 'SIMLIB_NREPEAT:   1 # use each LIBID for one time\n'
 'NGENTOT_LC:  13107\n'
 '\n'
 '# name of version to appear in $SNDATA_ROOT/SIM\n'
 'GENVERSION: MY_LSST_KN_6\n'
 '\n'
 '# define SED model and rate model\n'
 'GENMODEL:  '
 '$SNDATA_ROOT/models/SIMSED/SIMSED.KN-BULLA19/SIMSED.BULLA-BNS-M3-3COMP\n'
 'DNDZ: POWERLAW  1.0E-6  0.0  # volumetric rate = 1.0E-6/Mpc^3/yr (no '
 'z-dependence)\n'
 '\n'
 'GENFILTERS:   ugrizY\n'
 '\n'
 '# define distribution of KN parameters\n'
 'SIMSED_PARAM: COSTHETA\n'
 'GENPEAK_COSTHETA:  0.186585\n'
 'GENRANGE_COSTHETA: 0.0 1.0\n'
 'GENSIGMA_COSTHETA: 0.01 0.01\n'
 '\n'
 'SIMSED_PARAM: MEJDYN\n'
 'GENPEAK_MEJDYN:  0.003125\n'
 'GENRANGE_MEJDYN: 0.001 0.02\n'
 'GENSIGMA_MEJDYN: 0 0\n'
 '\n'
 'SIMSED_PARAM: MEJWIND\n'
 'GENPEAK_MEJWIND:  0.017428\n'
 'GENRANGE_MEJWIND: 0.01 0.13\n'
 'GENSIGMA_MEJWIND: 0 0\n'
 '\n'
 'SIMSED_PARAM: PHI\n'
 'GENPEAK_PHI:  22.447889\n'
 'GENRANGE_PHI: 0 90\n'
 'GENSIGMA_PHI: 1 1\n'
 '\n'
 '

In [ ]:
outfile = f"/fred/oz016/bgao_kn/data/SIM_INPUT/SIMGEN_KN_LSST_{sim_id}.INPUT"
with open(outfile, "w", encoding="utf-8") as f:
    f.write(text)